# Data Generation

Generate test questions from the credit risk knowledge base and run them through the agent to produce interaction logs for evaluation.

In [1]:
import sys
sys.path.insert(0, '../app')

import json
import random
from pathlib import Path
from pydantic import BaseModel
from pydantic_ai import Agent
from tqdm.auto import tqdm
from dotenv import load_dotenv

load_dotenv('../.env', override=True)

True

In [2]:
import ingest
import search_agent as sa
from logs import log_interaction_to_file

REPOS = [
    ('ing-bank', 'skorecard', 'main'),
    ('guillermo-navas-palencia', 'optbinning', 'master'),
    ('levist7', 'Credit_Risk_Modelling', 'main'),
]

print('Indexing repos...')
index = ingest.index_data(REPOS, chunk=True)
agent = sa.init_agent(index)
print('Agent ready.')

Indexing repos...
Agent ready.


In [3]:
# Sample documents from the index to base questions on
sample_docs = random.sample(index.docs, min(10, len(index.docs)))
print(f'Sampled {len(sample_docs)} documents')

Sampled 10 documents


In [4]:
question_generation_prompt = """
You are helping to create test questions for an AI agent that answers questions about credit risk scorecard development.

Based on the provided course content, generate realistic questions that practitioners might ask.

The questions should:
- Be natural and varied in style
- Range from simple to complex
- Include both specific technical questions and general credit risk questions

Generate one question for each record.
""".strip()

class QuestionsList(BaseModel):
    questions: list[str]

question_generator = Agent(
    name='question_generator',
    instructions=question_generation_prompt,
    model='gpt-4o-mini',
    output_type=QuestionsList
)

In [5]:
# Generate questions from sampled documents
docs_text = '\n\n---\n\n'.join(
    f"Document {i+1}:\n{doc.get('content', '')[:500]}"
    for i, doc in enumerate(sample_docs)
)

result = await question_generator.run(user_prompt=docs_text)
questions = result.output.questions

print(f'Generated {len(questions)} questions:')
for q in questions:
    print(f'  - {q}')

Generated 10 questions:
  - What is the significance of the AMT_ANNUITY variable in credit risk scorecard development?
  - Can you explain the output metrics of the EBM benchmark used for scoring credit risks?
  - How do I properly load and split credit card data for model training?
  - What are the advantages of using OptBinning for variable binning in a credit scorecard?
  - How can I manually define buckets in Skorecard, and what benefits does it provide?
  - What method can be used to sum the states when preparing data, and why is it essential?
  - What are the implications of using a bucketing process with and without redundant summary statistics in credit risk scorecards?
  - How would I generate counterfactual outcomes in a credit risk model, and what constraints should be considered?
  - How does the outlier detection process affect the binning results in OptBinning?
  - What function can be implemented to estimate the number of loan applications approved based on predicted pro

In [6]:
# Run agent on each question and log the interactions
LOG_DIR = Path('../app/logs')
LOG_DIR.mkdir(exist_ok=True)

for q in tqdm(questions):
    print(f'Q: {q}')
    result = await agent.run(user_prompt=q)
    print(f'A: {result.output[:200]}...')
    log_interaction_to_file(agent, result.new_messages(), source='ai-generated')
    print()

  0%|          | 0/10 [00:00<?, ?it/s]

Q: What is the significance of the AMT_ANNUITY variable in credit risk scorecard development?
A: The **AMT_ANNUITY** variable is an important feature in credit risk scorecard development. Here's a summary of its significance based on relevant information retrieved:

1. **Definition**: AMT_ANNUITY...

Q: Can you explain the output metrics of the EBM benchmark used for scoring credit risks?
A: The output metrics of the Explainable Boosting Machine (EBM) benchmark used for scoring credit risks generally include performance metrics that assess how well the model predicts borrower defaults and...

Q: How do I properly load and split credit card data for model training?
A: To properly load and split credit card data for model training, you can follow a structured pipeline that typically includes importing the necessary libraries, loading the dataset, preprocessing the d...

Q: What are the advantages of using OptBinning for variable binning in a credit scorecard?
A: Using OptBinning for vari

In [7]:
log_files = list(LOG_DIR.glob('credit_risk_agent_*.json'))
print(f'Total log files available for evaluation: {len(log_files)}')

Total log files available for evaluation: 16
